# non_sport_crypto metadata decision notebook

This notebook answers two selection questions for the non_sport_crypto market universe:

1. Should we keep only markets with cumulative volume_fp at least 10,000?
2. Should we keep only markets with a lifecycle of at least 5 days?

It also makes the temporal coverage of the downloaded market metadata explicit.

## Scope and interpretation

This is a market metadata snapshot, not daily candle coverage.

- volume_fp is the cumulative volume reported by the API, not daily volume.
- lifecycle duration is close_time minus open_time.
- date coverage below describes market lifecycle and retrieval dates; it does not prove that candles exist for every date.
- The notebook is read-only and analyzes only non_sport_crypto.

In [ ]:
from pathlib import Path
import json
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "db/kalshi_daily_probability_dataset.sqlite").exists()
    ),
    Path.cwd(),
)
DB_PATH = PROJECT_ROOT / "db/kalshi_daily_probability_dataset.sqlite"
SELECTION_ID = "20260817T124811Z-3b15c8a0-637b555156e3"
GROUPS = ["non_sport_crypto"]
VOLUME_THRESHOLD = 10_000
LIFETIME_THRESHOLD_DAYS = 5

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_PATH:", DB_PATH)
print("SELECTION_ID:", SELECTION_ID)
print("GROUPS:", GROUPS)

In [ ]:
def read_sql(query, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(query, conn, params=params)

def parse_json(value):
    try:
        return json.loads(value) if value else {}
    except (TypeError, json.JSONDecodeError):
        return {}

def run_group(config_json):
    config = parse_json(config_json)
    nested = config.get("config") or {}
    return config.get("selection_group") or nested.get("selection_group")

group_placeholders = ", ".join("?" for _ in GROUPS)
group_params = [SELECTION_ID, *GROUPS]

## 1. Snapshot inventory

In [ ]:
inventory = read_sql("""
    SELECT "raw_series" AS table_name, COUNT(*) AS rows FROM raw_series
    UNION ALL SELECT "raw_events", COUNT(*) FROM raw_events
    UNION ALL SELECT "raw_markets", COUNT(*) FROM raw_markets
    UNION ALL SELECT "raw_payloads", COUNT(*) FROM raw_payloads
""")
display(inventory)

runs = read_sql("""
    SELECT run_id, started_at_utc, finished_at_utc, status, source_mode,
           config_json, stats_json, error_text
    FROM metadata_runs
    ORDER BY started_at_utc DESC
""")
runs["selection_group"] = runs["config_json"].map(run_group)
display(runs[[
    "run_id", "selection_group", "source_mode", "status",
    "started_at_utc", "finished_at_utc"
]].head(15))

In [ ]:
markets = read_sql(f"""
    SELECT
        m.market_id, m.ticker, m.event_ticker,
        COALESCE(m.series_ticker, e.series_ticker) AS series_ticker,
        m.title, m.status, m.created_time, m.updated_time,
        m.open_time, m.close_time, m.settlement_ts,
        m.last_price_dollars, m.yes_bid_dollars, m.yes_ask_dollars,
        m.volume_fp, m.volume_24h_fp, m.open_interest_fp,
        m.liquidity_dollars, m.notional_value_dollars,
        m.mve_collection_ticker, m.result,
        m.source_endpoint, m.retrieved_at_utc, m.run_id,
        COALESCE(sm.selection_group, json_extract(r.config_json, '$.selection_group'))
            AS manifest_group,
        CASE
            WHEN m.series_ticker IS NOT NULL THEN 'market'
            WHEN e.series_ticker IS NOT NULL THEN 'event'
            ELSE 'unresolved'
        END AS series_mapping_source,
        CASE WHEN e.event_id IS NOT NULL THEN 1 ELSE 0 END
            AS event_metadata_available
    FROM raw_markets m
    LEFT JOIN raw_events e ON e.event_ticker = m.event_ticker
    LEFT JOIN series_selection_members sm
      ON sm.series_ticker = COALESCE(m.series_ticker, e.series_ticker)
     AND sm.selection_id = ?
     AND sm.eligible = 1
    LEFT JOIN metadata_runs r ON r.run_id = m.run_id
    WHERE COALESCE(sm.selection_group, json_extract(r.config_json, '$.selection_group'))
          IN ({group_placeholders})
    ORDER BY manifest_group, m.ticker
""", group_params)

print("Markets loaded:", len(markets))
display(markets.head(10))

In [ ]:
date_columns = [
    "created_time", "updated_time", "open_time",
    "close_time", "settlement_ts", "retrieved_at_utc",
]
numeric_columns = [
    "last_price_dollars", "yes_bid_dollars", "yes_ask_dollars",
    "volume_fp", "volume_24h_fp", "open_interest_fp",
    "liquidity_dollars", "notional_value_dollars",
]

for column in date_columns:
    markets[column] = pd.to_datetime(markets[column], utc=True, errors="coerce")
for column in numeric_columns:
    markets[column] = pd.to_numeric(markets[column], errors="coerce")

markets["series_ticker"] = markets["series_ticker"].replace("", pd.NA)
markets["lifetime_days"] = (
    markets["close_time"] - markets["open_time"]
).dt.total_seconds() / 86400

print("Market rows:", len(markets))
print("Unique events:", markets["event_ticker"].nunique())
print("Unique mapped series:", markets["series_ticker"].nunique())
display(markets[[
    "ticker", "event_ticker", "series_ticker", "status",
    "volume_fp", "open_time", "close_time", "lifetime_days"
]].head(10))

## 2. Data completeness and current coverage

In [ ]:
coverage = pd.DataFrame([{
    "markets": markets["ticker"].nunique(),
    "events": markets["event_ticker"].nunique(),
    "mapped_series": markets["series_ticker"].nunique(),
    "events_with_details": markets.loc[
        markets["event_metadata_available"].eq(1), "event_ticker"
    ].nunique(),
    "unresolved_series_mapping": markets["series_ticker"].isna().sum(),
    "missing_volume": markets["volume_fp"].isna().sum(),
    "missing_open_time": markets["open_time"].isna().sum(),
    "missing_close_time": markets["close_time"].isna().sum(),
    "missing_lifetime": markets["lifetime_days"].isna().sum(),
    "non_positive_lifetime": markets["lifetime_days"].le(0).sum(),
}])
coverage["event_detail_coverage_pct"] = (
    100 * coverage["events_with_details"] / coverage["events"].replace(0, np.nan)
)
display(coverage.T.rename(columns={0: "value"}))

retrieval_dates = (
    markets["retrieved_at_utc"].dt.floor("D").value_counts().sort_index()
    .rename("markets_retrieved")
    .to_frame()
)
display(Markdown("### Retrieval dates"))
display(retrieval_dates)

## 3. Decision question 1: volume_fp at least 10,000

In [ ]:
volume_quantiles = (
    markets["volume_fp"]
    .quantile([0, .25, .5, .75, .9, .95, .99, 1])
    .rename("volume_fp")
    .to_frame()
)
display(Markdown("### Volume distribution"))
display(volume_quantiles)

volume_thresholds = [0, 100, 1_000, 5_000, VOLUME_THRESHOLD, 25_000, 50_000, 100_000]
volume_sensitivity = pd.DataFrame([
    {
        "threshold_volume_fp": threshold,
        "markets": int(markets["volume_fp"].ge(threshold).sum()),
        "share_pct": 100 * markets["volume_fp"].ge(threshold).mean(),
        "events": markets.loc[markets["volume_fp"].ge(threshold), "event_ticker"].nunique(),
        "series": markets.loc[markets["volume_fp"].ge(threshold), "series_ticker"].nunique(),
    }
    for threshold in volume_thresholds
])
display(Markdown("### Retention sensitivity"))
display(volume_sensitivity.round({"share_pct": 2}))

volume_missing_or_zero = pd.DataFrame([{
    "missing_volume": markets["volume_fp"].isna().sum(),
    "zero_volume": markets["volume_fp"].eq(0).sum(),
    "positive_volume": markets["volume_fp"].gt(0).sum(),
    "kept_at_10000": markets["volume_fp"].ge(VOLUME_THRESHOLD).sum(),
    "share_kept_at_10000_pct": 100 * markets["volume_fp"].ge(VOLUME_THRESHOLD).mean(),
}])
display(volume_missing_or_zero)

## 4. Decision question 2: lifecycle at least 5 days

In [ ]:
lifetime_quantiles = (
    markets["lifetime_days"]
    .quantile([0, .25, .5, .75, .9, .95, .99, 1])
    .rename("lifetime_days")
    .to_frame()
)
display(Markdown("### Lifecycle distribution"))
display(lifetime_quantiles)

lifetime_thresholds = [0, 1, 3, LIFETIME_THRESHOLD_DAYS, 7, 14, 30, 90]
lifetime_sensitivity = pd.DataFrame([
    {
        "threshold_days": threshold,
        "markets": int(markets["lifetime_days"].ge(threshold).sum()),
        "share_pct": 100 * markets["lifetime_days"].ge(threshold).mean(),
        "events": markets.loc[markets["lifetime_days"].ge(threshold), "event_ticker"].nunique(),
        "series": markets.loc[markets["lifetime_days"].ge(threshold), "series_ticker"].nunique(),
    }
    for threshold in lifetime_thresholds
])
display(Markdown("### Retention sensitivity"))
display(lifetime_sensitivity.round({"share_pct": 2}))

lifetime_quality = pd.DataFrame([{
    "missing_lifetime": markets["lifetime_days"].isna().sum(),
    "zero_or_negative_lifetime": markets["lifetime_days"].le(0).sum(),
    "valid_positive_lifetime": markets["lifetime_days"].gt(0).sum(),
    "kept_at_5_days": markets["lifetime_days"].ge(LIFETIME_THRESHOLD_DAYS).sum(),
    "share_kept_at_5_days_pct": 100 * markets["lifetime_days"].ge(LIFETIME_THRESHOLD_DAYS).mean(),
}])
display(lifetime_quality)

## 5. Combined filter decision

In [ ]:
def summarize_subset(name, mask):
    subset = markets.loc[mask]
    return {
        "scenario": name,
        "markets": len(subset),
        "share_of_markets_pct": 100 * len(subset) / max(len(markets), 1),
        "events": subset["event_ticker"].nunique(),
        "mapped_series": subset["series_ticker"].nunique(),
        "median_volume_fp": subset["volume_fp"].median(),
        "median_lifetime_days": subset["lifetime_days"].median(),
        "missing_event_details": subset.loc[
            subset["event_metadata_available"].eq(0), "event_ticker"
        ].nunique(),
    }

volume_mask = markets["volume_fp"].ge(VOLUME_THRESHOLD)
lifetime_mask = markets["lifetime_days"].ge(LIFETIME_THRESHOLD_DAYS)

decision_table = pd.DataFrame([
    summarize_subset("all downloaded", pd.Series(True, index=markets.index)),
    summarize_subset("volume_fp >= 10,000", volume_mask),
    summarize_subset("lifetime_days >= 5", lifetime_mask),
    summarize_subset("both filters", volume_mask & lifetime_mask),
])
display(decision_table.round({
    "share_of_markets_pct": 2,
    "median_volume_fp": 2,
    "median_lifetime_days": 2,
}))

display(Markdown(
    "The key research trade-off is visible in the share_of_markets_pct column: "
    "a threshold is useful only if the retained universe is still broad enough "
    "and the discarded markets are mostly low-information markets rather than "
    "entirely removing important event types."
))

## 6. Which dates are represented?

In [ ]:
lifecycle = markets.dropna(subset=["open_time", "close_time"]).copy()
lifecycle["open_date"] = lifecycle["open_time"].dt.normalize().dt.tz_localize(None)
lifecycle["close_date"] = lifecycle["close_time"].dt.normalize().dt.tz_localize(None)

date_range_summary = pd.DataFrame([{
    "earliest_open_date": lifecycle["open_date"].min(),
    "latest_open_date": lifecycle["open_date"].max(),
    "earliest_close_date": lifecycle["close_date"].min(),
    "latest_close_date": lifecycle["close_date"].max(),
    "markets_with_valid_intervals": len(lifecycle),
}])
display(Markdown("### Overall lifecycle range"))
display(date_range_summary)

open_counts = lifecycle["open_date"].value_counts().rename("markets_opening").to_frame()
close_counts = lifecycle["close_date"].value_counts().rename("markets_closing").to_frame()
display(Markdown("### Dates with the most market openings"))
display(open_counts.head(20))
display(Markdown("### Dates with the most market closings"))
display(close_counts.head(20))

start_events = pd.Series(1, index=lifecycle["open_date"])
end_events = pd.Series(-1, index=lifecycle["close_date"] + pd.Timedelta(days=1))
daily_delta = pd.concat([start_events, end_events]).groupby(level=0).sum().sort_index()
full_index = pd.date_range(daily_delta.index.min(), daily_delta.index.max(), freq="D")
active_by_day = daily_delta.reindex(full_index, fill_value=0).cumsum().rename("active_markets")
active_by_day.index.name = "date"

display(Markdown(
    "### Dates with the largest lifecycle-based active-market counts"
))
display(active_by_day.sort_values(ascending=False).head(20).to_frame())

monthly_active = active_by_day.resample("MS").max().rename("max_active_markets").to_frame()
display(Markdown("### Monthly lifecycle coverage"))
display(monthly_active.sort_values("max_active_markets", ascending=False).head(24))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if not retrieval_dates.empty:
    retrieval_dates.plot.bar(ax=axes[0], legend=False, color="steelblue")
axes[0].set_title("Markets retrieved by UTC date")
axes[0].set_xlabel("retrieval date")
axes[0].set_ylabel("market rows")
axes[0].tick_params(axis="x", rotation=45)

if not monthly_active.empty:
    monthly_active.plot(ax=axes[1], legend=False, color="darkorange")
axes[1].set_title("Maximum active markets by lifecycle month")
axes[1].set_xlabel("month")
axes[1].set_ylabel("active market rows")
axes[1].grid(alpha=.2)

plt.tight_layout()
plt.show()

## Out of scope for this decision notebook

The following diagnostics were removed from the main notebook because they do not answer the two selection questions:

- market_type and strike_type breakdowns;
- price spread and price-level diagnostics;
- liquidity and open-interest distributions;
- top-market and top-event lists;
- detailed event-category exploration.

They can be reintroduced later if the candle analysis shows a specific data-quality problem. For now, the decision should be based on retention, lifecycle validity, date representation, and eventual daily-candle coverage.